# Shared representations of educational data

Prepare MoocRadar, ALPACAS, and Moodle inputs, run the experiment, and inspect its results. Launch this notebook from the repository root with `jupyter lab experiment.ipynb`.

The default `smoke` mode checks execution. Use `measure` for the complete protocol. Set `RESULTS` to an existing experiment directory to skip preparation and measurement and analyze that run.


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display, Markdown

from scripts import prepare_moocradar, prepare_alpacas, prepare_moodle, prepare_serving
from scripts.prepare_moocradar import CUTS, SERVING_PARQUETS, SERVING_ARRAYS
from experiments.run import run_experiment
from experiments.verify import verify


In [ ]:
MODE = 'smoke'
RESULTS = None  # For existing results: Path('outputs/measure_...').
CONNECTION_FILE = Path('.env')

MOOCRADAR_RAW = Path('local_data/moocradar')
ALPACAS_RAW = Path('local_data/alpacas')
MOODLE_ZIP = Path('local_data/moodle.zip')
MOODLE_RAW = Path('local_data/moodle')


## Prepare the data

Download `student-problem-middle.json` and `problem.json` from the [MoocRadar repository](https://github.com/THU-KEG/MOOC-Radar), and the version 1 [Moodle ZIP](https://doi.org/10.17632/x8ntc8skgs.1). The ALPACAS function downloads the pinned source release. Completed preparation outputs are reused. Partial outputs require inspection before retrying.


In [ ]:
def needs_preparation(required_files):
    if RESULTS is not None:
        return False
    if all(path.is_file() for path in required_files):
        return False
    if any(path.exists() for path in required_files):
        raise FileExistsError('Partial prepared inputs exist. Inspect their directory before retrying.')
    return True


In [ ]:
moocradar = Path('data/moocradar')
required = [moocradar / name for name in
            ['events.parquet', 'problems.parquet', 'history.parquet', 'preparation_manifest.json']]
required += [moocradar / 'serving' / cut / name
             for cut in CUTS for name in SERVING_PARQUETS + SERVING_ARRAYS]
required += [moocradar / 'workloads' / f'{kind}_{cut}.parquet'
             for cut in CUTS for kind in ['requests', 'journal']]
if needs_preparation(required):
    moocradar_report = prepare_moocradar(
        raw_dir=MOOCRADAR_RAW,
        out_dir=moocradar,
        serving_out_dir=moocradar / 'serving',
        workloads_out_dir=moocradar / 'workloads',
    )


In [ ]:
alpacas = Path('data/alpacas/training')
required = [alpacas / f'{kind}_{period}.parquet'
            for kind in ['events', 'participants'] for period in ['1819', '1920', '2021']]
required.append(alpacas / 'input_audit.json')
if needs_preparation(required):
    alpacas_report = prepare_alpacas(raw_dir=ALPACAS_RAW, out_dir=alpacas)


In [ ]:
moodle = Path('data/moodle/training')
required = [moodle / f'{kind}_{split}.parquet'
            for kind in ['events', 'participants'] for split in ['train', 'calibration', 'test']]
required += [moodle / name for name in ['channel_vocabulary.json', 'input_audit.json']]
if needs_preparation(required):
    moodle_report = prepare_moodle(
        source_zip=MOODLE_ZIP,
        raw_dir=MOODLE_RAW,
        out_dir=moodle,
        provenance_dir=Path('outputs/moodle_preparation'),
    )


In [ ]:
for dataset in ['alpacas', 'moodle']:
    serving = Path('data') / dataset / 'serving'
    required = [serving / name for name in ['raw.parquet', 'participants.parquet',
                'serving_population.json', 'channel_vocabulary.json', 'preparation.json']]
    if needs_preparation(required):
        prepare_serving(dataset, training_dir=Path('data') / dataset / 'training', out_dir=serving)


## Run the experiment

Set the ClickHouse credentials in `.env` using `.env.example` and start the server before this cell. The bundled service starts with `docker compose up -d` from the repository root.

`run_experiment` executes the measurement stages in fresh worker processes, verifies their outputs, and saves the summaries. Each new run receives its own output directory. Run one experiment at a time.


In [ ]:
if RESULTS is None:
    RESULTS = run_experiment(mode=MODE, connection_file=CONNECTION_FILE)
else:
    RESULTS = Path(RESULTS)
    verification = verify(RESULTS)

print('Results:', RESULTS)


## Results

Positive reduction means that the variant uses less storage or time than its baseline. Timing estimates use paired arithmetic means and 95% moving-block bootstrap intervals. An interval spanning zero does not establish a timing benefit for that comparison.


In [ ]:
state = json.loads((RESULTS / 'campaign.json').read_text(encoding='utf-8'))
if state['status'] != 'completed':
    raise ValueError('The experiment is incomplete. Inspect its logs before analyzing it.')
summary = json.loads((RESULTS / 'summary.json').read_text(encoding='utf-8'))
if summary['smoke']:
    display(Markdown('**Smoke check: these are shortened execution checks, not full-experiment estimates.**'))
else:
    display(Markdown('**Complete experiment.**'))

timings = pd.DataFrame(summary['comparisons'])
storage = pd.DataFrame(summary['storage'])

def timing_table(frame, context, baseline_label='Baseline (ms)', variant_label='Variant (ms)'):
    table = frame.copy()
    table['95% CI (%)'] = table.ci95_reduction_percent.map(
        lambda interval: f'{interval[0]:.1f} to {interval[1]:.1f}')
    columns = context + ['baseline_mean_ms', 'variant_mean_ms', 'reduction_percent',
                         '95% CI (%)', 'n_pairs']
    return table[columns].rename(columns={
        'baseline_mean_ms': baseline_label, 'variant_mean_ms': variant_label,
        'reduction_percent': 'Reduction (%)', 'n_pairs': 'Paired observations',
    }).round(2).reset_index(drop=True)


## Shared versus separate representations

In [ ]:
common = timings.loc[timings.module.eq('common')].copy()
common['measure'] = common.phase.map({'answer': 'Answer', 'build': 'Construction', 'update': 'Update'})
display(timing_table(common, ['dataset', 'measure'], 'Separate (ms)', 'Shared (ms)'))

sizes = storage.loc[storage.module.eq('common')].copy()
sizes['Separate (MiB)'] = sizes.baseline_mean_bytes / 1024**2
sizes['Shared (MiB)'] = sizes.variant_mean_bytes / 1024**2
display(sizes[['dataset', 'Separate (MiB)', 'Shared (MiB)', 'reduction_percent']]
        .rename(columns={'reduction_percent': 'Reduction (%)'}).round(2).reset_index(drop=True))


## Catalog selection

In [ ]:
display(timing_table(timings.loc[timings.module.eq('selection')],
                     ['dataset', 'baseline', 'variant']))


## Scaling

Select the history or served-population scenario and a baseline/variant pair. All comparisons remain available in `timings`.


In [ ]:
SCENARIO = 'history'
BASELINE = 'separate_selected'
VARIANT = 'shared_selected'

scaling = timings.loc[timings.module.eq('scaling') & timings.scenario.eq(SCENARIO)
                      & timings.baseline.eq(BASELINE) & timings.variant.eq(VARIANT)]
if scaling.empty:
    raise ValueError('No scaling comparisons match these parameters.')
display(timing_table(scaling, ['scenario', 'factor', 'fraction', 'phase']))

## Correction processing

Policies: `B` fully recomputes the state, `BB` uses batch bounds, `S` processes selected changes, and `A` selects adaptively. Contracts require decisions (`D`), decisions and monitoring (`DM`), or numerical predictions and monitoring (`PM`). Answer time and the total time including completion of the exact state are reported separately.

In [ ]:
MAINTENANCE_BASELINE = 'B'
MAINTENANCE_VARIANT = 'A'

maintenance = timings.loc[timings.module.eq('maintenance')
                          & timings.baseline.eq(MAINTENANCE_BASELINE)
                          & timings.variant.eq(MAINTENANCE_VARIANT)]
display(timing_table(maintenance, ['factor', 'contract', 'phase']))

## Bootstrap sensitivity

In [ ]:
sensitivity = []
for row in common.to_dict('records'):
    for block, interval in row['intervals_by_block_length'].items():
        lower, upper = interval['ci95_reduction_percent']
        sensitivity.append({'dataset': row['dataset'], 'measure': row['measure'],
                            'block_length': int(block), 'lower_95_percent': lower,
                            'upper_95_percent': upper})
display(pd.DataFrame(sensitivity).round(2))